In [1]:
"""
Joint training vs Bagging — bias reduction smoke test
=======================================================

가설: capacity(개별 net param 수)를 고정한 채, k개 net을 joint loss로
학습시키면(각자 독립 학습 후 평균이 아니라, ensemble output에 직접 loss를 걸면)
bagging보다 낮은 bias(=낮은 fit error)를 얻을 수 있다.
(단순 variance reduction이 아니라, boosting-like division of labor가
capacity 증가 없이 collective capacity를 늘리는 것인지 확인)

Setup:
- N개의 random target vector (구조 없는 target -> 순수 memorization,
  즉 이 문제의 "bias"는 순전히 capacity 부족에서만 옴)
- 개별 net: input = one-hot(index), hidden=H (일부러 작게 -> underparam 강제)
- 3-way 비교: (1) single net, (2) bagging (k개 독립 학습 후 평균),
  (3) joint (k개를 ensemble-output loss로 동시 학습)
- Diagnostic: joint 학습 후 각 net이 서로 다른 index 부분집합에서
  더 정확한지 (division of labor 실제로 생겼는지) 확인
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# ---------------- Config ----------------
N = 200          # memorize할 random vector 개수
D = 16           # target vector 차원
H = 8            # 개별 net hidden dim (일부러 작게 -> underparam)
K = 5            # ensemble member 수
STEPS = 3000
LR = 1e-2
DEVICE = "cpu"

# ---------------- Data ----------------
# target[i] = 고정된 random vector (구조 없음 -> 순수 memorization 문제)
targets = torch.randn(N, D)


class TinyNet(nn.Module):
    """input: one-hot(N) -> output: R^D. Capacity를 H로 고정."""
    def __init__(self, n_in, hidden, d_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


def make_inputs():
    return torch.eye(N)  # one-hot encoding of index


def mse_per_index(pred, target):
    # (N, D) -> (N,) per-sample MSE, for division-of-labor diagnostic
    return ((pred - target) ** 2).mean(dim=1)


# ---------------- (1) Single net baseline ----------------
def train_single():
    x = make_inputs()
    net = TinyNet(N, H, D)
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    for step in range(STEPS):
        opt.zero_grad()
        pred = net(x)
        loss = ((pred - targets) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        final_pred = net(x)
        final_loss = ((final_pred - targets) ** 2).mean().item()
        per_idx = mse_per_index(final_pred, targets)
    return final_loss, per_idx, [net]


# ---------------- (2) Bagging: independent training, then average ----------------
def train_bagging(k=K):
    x = make_inputs()
    nets = [TinyNet(N, H, D) for _ in range(k)]
    opts = [torch.optim.Adam(net.parameters(), lr=LR) for net in nets]

    for step in range(STEPS):
        for net, opt in zip(nets, opts):
            opt.zero_grad()
            pred = net(x)
            # 각자 독립적으로 "이 target을 나 혼자 맞춰라"
            loss = ((pred - targets) ** 2).mean()
            loss.backward()
            opt.step()

    with torch.no_grad():
        preds = torch.stack([net(x) for net in nets], dim=0)  # (k, N, D)
        ensemble_pred = preds.mean(dim=0)
        final_loss = ((ensemble_pred - targets) ** 2).mean().item()
        per_idx = mse_per_index(ensemble_pred, targets)
        per_net_per_idx = torch.stack(
            [mse_per_index(preds[i], targets) for i in range(k)], dim=0
        )  # (k, N)
    return final_loss, per_idx, nets, per_net_per_idx


# ---------------- (3) Joint training: loss on ensemble output directly ----------------
def train_joint(k=K):
    x = make_inputs()
    nets = [TinyNet(N, H, D) for _ in range(k)]
    params = [p for net in nets for p in net.parameters()]
    opt = torch.optim.Adam(params, lr=LR)

    for step in range(STEPS):
        opt.zero_grad()
        preds = torch.stack([net(x) for net in nets], dim=0)  # (k, N, D)
        ensemble_pred = preds.mean(dim=0)
        # ensemble output에 직접 loss -> 각 net이 서로의 잔차를 메꾸도록 압력
        loss = ((ensemble_pred - targets) ** 2).mean()
        loss.backward()
        opt.step()

    with torch.no_grad():
        preds = torch.stack([net(x) for net in nets], dim=0)
        ensemble_pred = preds.mean(dim=0)
        final_loss = ((ensemble_pred - targets) ** 2).mean().item()
        per_idx = mse_per_index(ensemble_pred, targets)
        per_net_per_idx = torch.stack(
            [mse_per_index(preds[i], targets) for i in range(k)], dim=0
        )  # (k, N)
    return final_loss, per_idx, nets, per_net_per_idx


# ---------------- Division-of-labor diagnostic ----------------
def division_of_labor_score(per_net_per_idx):
    """
    각 index에서 '어느 net이 제일 잘 맞추는가'를 보고,
    그 best-net assignment가 net들 사이에 고르게 분산되어 있는지 측정.
    완전히 division of labor가 없으면(모든 net이 비슷비슷하면) best-net이
    거의 랜덤하게 흩어지거나 특정 net에 쏠릴 것.
    Division of labor가 있으면, 각 net이 서로 다른 index cluster에서
    확실하게 우위를 갖는 구조가 나타나야 함 -> best-net 분포의 entropy가
    "각자 확실한 담당 구역이 있다"는 방향으로 나타나는지 확인.
    """
    k, n = per_net_per_idx.shape
    best_net = per_net_per_idx.argmin(dim=0)  # (N,) which net is best per index
    counts = torch.bincount(best_net, minlength=k).float()
    frac = counts / counts.sum()
    # 얼마나 고르게 분산되어 있는지 (uniform=1/k면 이상적인 균등 분업)
    uniform = torch.full_like(frac, 1.0 / k)
    l1_dev_from_uniform = (frac - uniform).abs().sum().item()
    return frac.tolist(), l1_dev_from_uniform


# ---------------- Run ----------------
if __name__ == "__main__":
    print(f"Config: N={N} targets, D={D} dim, H={H} hidden (per-net), K={K} members\n")

    single_loss, single_per_idx, _ = train_single()
    print(f"[1] Single net (baseline, capacity=1xH):        final MSE = {single_loss:.5f}")

    bag_loss, bag_per_idx, bag_nets, bag_per_net_per_idx = train_bagging()
    print(f"[2] Bagging (k={K}, independent + average):      final MSE = {bag_loss:.5f}")

    joint_loss, joint_per_idx, joint_nets, joint_per_net_per_idx = train_joint()
    print(f"[3] Joint training (k={K}, ensemble-output loss): final MSE = {joint_loss:.5f}\n")

    print("--- Bias reduction check ---")
    print(f"Bagging vs single:  {(1 - bag_loss/single_loss)*100:+.1f}% change")
    print(f"Joint   vs single:  {(1 - joint_loss/single_loss)*100:+.1f}% change")
    print(f"Joint   vs bagging: {(1 - joint_loss/bag_loss)*100:+.1f}% change\n")

    print("--- Division of labor diagnostic ---")
    bag_frac, bag_dev = division_of_labor_score(bag_per_net_per_idx)
    joint_frac, joint_dev = division_of_labor_score(joint_per_net_per_idx)
    print(f"Bagging  best-net fraction per member: {[f'{f:.2f}' for f in bag_frac]}  (L1 dev from uniform: {bag_dev:.3f})")
    print(f"Joint    best-net fraction per member: {[f'{f:.2f}' for f in joint_frac]}  (L1 dev from uniform: {joint_dev:.3f})")
    print("\n(해석: joint의 L1 dev가 uniform에 가깝고 bagging보다 뚜렷한 구조를 보이면")
    print(" division-of-labor가 실제로 생겼다는 신호. 반대로 특정 net에 쏠려있으면")
    print(" 그냥 '제일 잘 초기화된 net이 다 먹는' 현상이라 division of labor가 아님.)")

Config: N=200 targets, D=16 dim, H=8 hidden (per-net), K=5 members

[1] Single net (baseline, capacity=1xH):        final MSE = 0.37982
[2] Bagging (k=5, independent + average):      final MSE = 0.37957
[3] Joint training (k=5, ensemble-output loss): final MSE = 0.00002

--- Bias reduction check ---
Bagging vs single:  +0.1% change
Joint   vs single:  +100.0% change
Joint   vs bagging: +100.0% change

--- Division of labor diagnostic ---
Bagging  best-net fraction per member: ['0.06', '0.19', '0.14', '0.14', '0.47']  (L1 dev from uniform: 0.540)
Joint    best-net fraction per member: ['0.11', '0.26', '0.31', '0.25', '0.08']  (L1 dev from uniform: 0.430)

(해석: joint의 L1 dev가 uniform에 가깝고 bagging보다 뚜렷한 구조를 보이면
 division-of-labor가 실제로 생겼다는 신호. 반대로 특정 net에 쏠려있으면
 그냥 '제일 잘 초기화된 net이 다 먹는' 현상이라 division of labor가 아님.)
